In [12]:
import pandas as pd

In [13]:
df = pd.read_csv('jobs_dataset_with_features.csv')

In [14]:
df.shape

(1615940, 2)

In [15]:
df.head()

,Role,Features
0,Social Media Manager,5 to 15 Years Digital Marketing Specialist M.T...
1,Frontend Web Developer,"2 to 12 Years Web Developer BCA HTML, CSS, Jav..."
2,Quality Control Manager,0 to 12 Years Operations Manager PhD Quality c...
3,Wireless Network Engineer,4 to 11 Years Network Engineer PhD Wireless ne...
4,Conference Manager,1 to 12 Years Event Manager MBA Event planning...


In [16]:
df['Role'].value_counts()

Role
Interaction Designer            20580
Network Administrator           17470
User Interface Designer         14036
Social Media Manager            13945
User Experience Designer        13935
                                ...  
Inventory Control Specialist     3342
Budget Analyst                   3335
Clinical Nurse Manager           3324
Social Science Researcher        3321
Paid Advertising Specialist      3306
Name: count, Length: 376, dtype: int64

In [17]:
# Dropping classes with less than 6500 instances

min_count = 6500
role_counts = df['Role'].value_counts() 
dropped_classes = role_counts [role_counts < min_count].index
filtered_df = df [~df['Role'].isin (dropped_classes)].reset_index(drop=True)

# Checking the updated role counts
filtered_df['Role'].value_counts()

Role
Interaction Designer          20580
Network Administrator         17470
User Interface Designer       14036
Social Media Manager          13945
User Experience Designer      13935
                              ...  
Benefits Coordinator           6839
Research Analyst               6830
Administrative Coordinator     6803
IT Support Specialist          6799
UI/UX Designer                 6743
Name: count, Length: 61, dtype: int64

In [18]:
len(filtered_df ['Role'].value_counts())

61

In [19]:
df = filtered_df.sample(n=10000)
df.head()

,Role,Features
163424,Backend Developer,2 to 8 Years Software Engineer PhD Proficiency...
202759,Supply Chain Manager,2 to 12 Years Operations Manager BA Supply cha...
268682,Market Research Analyst,3 to 8 Years Research Analyst B.Tech Market re...
349373,Client Relationship Manager,2 to 12 Years Account Manager B.Com Client rel...
150393,Water Resources Engineer,2 to 13 Years Civil Engineer B.Tech Water reso...


# TF-IDF

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Splitting the data into features (X) and target (y)
X = df['Features']
y = df['Role']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [23]:
# Random ForestClassifier
rf_classifier = RandomForestClassifier()
rf_classifier.fit(X_train_tfidf, y_train)

# Predictions
y_pred = rf_classifier.predict(X_test_tfidf)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 1.0


# Recommendation

In [26]:
# clean text

import re
import string

def cleanResume(txt):
    cleanText = re.sub(r'http\S+\s', '', txt)
    cleanText = re.sub(r'RT|cc', ' ', cleanText)
    cleanText = re.sub(r'#\S+\s', ' ', cleanText)
    cleanText = re.sub(r'@\S+', ' ', cleanText)
    cleanText = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', cleanText)
    cleanText = re.sub(r'[^\x00-\x7f]', ' ', cleanText)
    cleanText = re.sub(r'\s+', ' ', cleanText)
    return cleanText


# Prediction and Category Name
def job_recommendation(resume_text):
    resume_text = cleanResume(resume_text)
    resume_tfidf = tfidf_vectorizer.transform([resume_text])
    predicted_category = rf_classifier.predict(resume_tfidf)[0]
    return predicted_category


In [27]:
# Example Usage

resume_file = """objective:
A creative and detail-oriented Designer with a passion for visual communication and brand identity seek

Education:
-Bachelor of Fine Arts in Graphic Design, XYZ College, GPA: 3.7/4.0
-Diploma in Web Design, ABC Institute, GPA: 3.9/4.0

Skills:
-Proficient in Adobe Creative Suite (Photoshop, Illustrator, InDesign)
-Strong understanding of typography, layout, and color theory
-Experience in both print and digital design
-Ability to conceptualize and execute design projects from concept to completion
-Excellent attention to detail and time management skills

Experience:
Graphic Designer | XYZ Design Studio
-Created visually appealing graphics for various marketing materials, including brochures, flyers, anc Collab…

Projects:
Rebranding Campaign for XYZ Company: Led a team to redesign the company's logo, website, and marketir

Packaging Design for ABC Product Launch: Developed eye-catching packaging designs for a new product

Certifications:
- Adobe Certified Expert (ACE) in Adobe Illustrator
Responsive Web Design Certification from Udemy

Languages:
English (Native)
Spanish (Intermediate)
"""

predicted_category = job_recommendation(resume_file)
print("Predicted Category:", predicted_category)


Predicted Category: User Interface Designer


In [28]:
# Example Usage

resume_file = """Objective:

Dedicated and results-oriented Banking professional with a strong background in financial analysis and

Education:

Bachelor of Business Administration in Finance, XYZ University, GPA: 3.8/4.0

Certified Financial Analyst (CFA) Level I Candidate

Skills:

Proficient in financial modeling and analysis using Excel, Bloomberg Terminal, and other financial sc

Extensive knowledge of banking products and services, including loans, mortgages, and investment proc

Strong understanding of regulatory compliance and risk management practices in the banking industry

Excellent communication and interpersonal skills, with a focus on building rapport with clients and c

Ability to work efficiently under pressure and adapt to …

Internship | GHI Investments

Assisted portfolio managers with investment research and analysis, including industry and company-spe

Prepared investment presentations and reports for clients, highlighting investment opportunities

Conducted market research and analysis to identify trends and opportunities in the financial markets

Certifications:

Certified Financial Planner (CFP)

Series 7 and Series 63 Securities Licenses

Languages:

English (Native)

Spanish (Proficient)
"""

predicted_category = job_recommendation(resume_file)
print("Predicted Category:", predicted_category)


Predicted Category: Financial Analyst


In [32]:
import os
import pickle

os.makedirs("models", exist_ok=True)
pickle.dump(rf_classifier, open('models/rf_classifier_job_recommendation.pkl','wb'))
pickle.dump(tfidf_vectorizer, open('models/tfidf_vectorizer_job_recommendation.pkl','wb'))

In [31]:
import os
import pickle

os.makedirs("models", exist_ok=True)

pickle.dump(
    rf_classifier_job_recommendation,
    open("models/rf_classifier_job_recommendation.pkl", "wb")
)


NameError: name 'rf_classifier_job_recommendation' is not defined